## Ranked Retrieval
**Nel Boolean Retrieval un documento o soddisfa la query oppure no, non esistono gradi di rilevanza**. Questo approccio è ok per utenti esperti, in grado di formulare query precise, oppure per applicazioni automatiche in grado di gestire anche migliaia di risultati. In generale però non è l'ideale per utenti medi, dal momento che non sanno scrivere query efficaci e non riescono a filtrare migliaia di documenti (oppure semplicemente non vogliono farlo). 

Da qui nasce il problema **feast or famine**: con il modello booleano una query può restituire zero documenti (famine) oppure migliaia di documenti (feast). Ad esempio una query generica come "world climate crisis" potrebbe restituire migliaia di documenti, mentre una query più specifica come "world climate crisis merkel" potrebbe restituirne zero. Questo succede perché gli operatori booleani sono rigidi: AND tende a restringere troppo i risultati, mentre OR tende a espanderli troppo.

Si vuole quindi risolvere il problema introducendo un modello di **ranked retrieval**, in cui i documenti che potrebbero soddisfare una query vengono ordinati in base a un punteggio di rilevanza. In questo modo si risolve il problema del feast in quanto anche se una query è generica, saranno mostrati solo i primi k documenti più rilevanti. Riguardo famine invece, come vedremo, si introduce una misura di similarità che permette di rilassare la rigidità degli operatori booleani, restituendo comunque documenti che sono simili alla query anche se non la soddisfano esattamente.

Per implementare un sistema di ranked retrieval è anzitutto necessario che ad ogni coppia query-documento sia associato un punteggio di rilevanza. Questo punteggio viene calcolato tramite una funzione di similarità, che intuitivamente dovrebbe restituire uno score pari a:
- 0 se il documento è completamente irrilevante per la query
- 1 se il documento è perfettamente rilevante per la query
- un valore intermedio se il documento è parzialmente rilevante per la query

### Jaccard Similarity
Una prima intuizione per una funzione di similarità è quella di usare la Jaccard Similarity, per misurare la similarità tra i due insiemi di termini (quello della query e quello del documento). 
$$ J(Q, D) = \frac{|Q \cap D|}{|Q \cup D|} $$
Applicato a query e documento, la Jaccard misura quindi quanti termini **distinti** i due condividono rispetto al totale dei termini **distinti** presenti. Ex. tra query "idles of March" e il doc "Caesar died in March" l'unico termine comune è March e il numero totale di termini distinti è 6 -> Jaccard = 1/6.

Tuttavia la Jaccard Similarity presenta limiti evidenti:
- **non considera la term frequency**: se un termine appare più volte in un documento, questo non aumenta la sua rilevanza rispetto alla query, dal momento che si confrontano solo insiemi di termini distinti
- **non è in grado di distinguere termini rari e termini frequenti** (e i termini rari sono molto informativi per la rilevanza di un documento rispetto a una query)
- **non gestisce bene le differenze di lunghezza tra query e documento**: es. query: "car climate", doc1: "car climate" e doc2: "car climate change and its impact on the environment". La Jaccard similarity tra query e doc1 è 1, mentre tra query e doc2 è 2/9, nonostante doc2 contenga tutti i termini della query.

### TF-IDF
Per superare i limiti della Jaccard Similarity, si introduce la misura di similarità basata su TF-IDF (Term Frequency - Inverse Document Frequency). 

Partiamo dal caso semplice di una query con un solo termine t. Se t non appare nel documento -> lo score deve essere 0. Se t appare -> lo score aumenta. Set appare più volte -> lo score aumenta ancora di più. Per calcolare questo punteggio, abbiamo bisogno di rappresentare i documenti in modo diverso rispetto alla Jaccard Similarity (non più come semplici insiemi di termini distinti).

La prima idea in questo senso è una **Binary Incidence Matrix**: una rappresentazione **densa** in cui ogni riga rappresenta un termine mentre ogni colonna un documento. In posizione (i, j) c'è un 1 se il termine i appare nel documento j, altrimenti c'è uno 0 -> ogni documento è rappresentato come vettore binario.

Tuttavia bisogna come detto anche tener conto della frequenza dei termini per capire quanto un termine è importante per un documento. Per questo si utilizza una **Count Matrix**: stessa identica matrice di prima, ma invece di 1 e 0, in posizione (i, j) c'è il numero di volte che il termine i appare nel documento j. In questo modo ogni documento è rappresentato come un vettore di frequenze dei termini. Un documento non è più un vettore binario, ma un count vector.

<img src=img/cv.png width=400>

Come già anticipato nelle lezioni precedenti, questo è un **Bag of Words** model: non si tiene affatto conto dell'ordine dei termini, ma solo della loro frequenza -> in questo modo il problema diventa più semplice da trattare matematicamente, ma si perde informazione importante.

#### TF (Term Frequency)
A questo punto entra in gioco la **term frequencty tf**:
$$ tf_{t, d} = \text{numero di volte in cui il termine t appare nel documento d} $$
rappresenta null'altro che l'informazione contenuta nella count matrix. Tuttavia usare la raw frequency non è l'ideale: se un termine appare 10 volte in un documento e in un altro una sola volta, ciò non significa necessariamente che il primo documento è 10 volte più rilevante della query rispetto al secondo, **la rilevanza non cresce proporzionalmente con il numero di occorrenze**.

Intuitivamente sono **le prime occorrenze di un termine ad essere davvero informative**: da 0 a 1 è importantissimo, da 1 a 2 utile, da 100 a 101 contribuisce di pochissimo. Per correggere il problema quindi si usa il **log frequency weighting**:
$$ w_{t, d} = \begin{cases} 1 + \log_{10}(tf_{t, d}) & \text{se } tf_{t, d} > 0 \\ 0 & \text{altrimenti} \end{cases} $$
In questo modo si attenua l'effetto delle frequenze elevate: se tf 0 allora w 0, se tf 1 allora w 1, se tf 10 allora w 2, se tf 1000 allora w 4.

Infine, dopo aver definito questi pesi, lo score query-documento è dato dalla somma di tutti i contributi dei termini comuni tra query e documento:
$$ \text{tf-matching-score}(q, d) = \sum_{t \in q \cap d} (1 + \log(tf_{t,d})) $$

ex. query = "information on cars", doc = "all you've ever wanted do know about cars". Calcolo tf-matching-score: i termini in comune sono solo "cars" -> 1+log(1) = 1. 

#### IDF (Inverse Document Frequency)
Il tf da solo non basta: ci sono parole che compaiono spesso ovunque (stop words) e quindi non sono affatto informative per la rilevanza di un documento rispetto a una query. **Al contrario parole rare nella collezione sono molto informative**: ad esempio "Phelathylamine" appare solo in pochi documenti, e se cerco la parola è molto probabile che quei documenti siano effettivamente pertinenti. **Per tale ragione vogliamo dare peso maggiore ai termini rrari e peso minore ai termini frequenti**.

Per fare questo si introduce l'**inverse document frequency idf**:
$$ idf_t = \log_{10} \left( \frac{N}{df_t} \right) $$
dove N è il numero totale di documenti nella collezione e df_t è il numero di documenti in cui appare il termine t. Usando idf_t, se t appare in pochi documenti -> df_t è piccolo -> idf_t (peso) è grande (t è molto informativo). Se t appare in molti documenti -> df_t è grande -> idf_t è piccolo (t è poco informativo).

**Perché il logaritmo**? Di nuovo, perché se un termine appare in 10 documenti è più informativo di un termine che appare in 1000 documenti, ma non è 100 volte più informativo. Si vogliono inoltre evitare valori di idf troppo elevati e restare nella scala di tf per poterli combinare nella misura finale. Es. con N = 1.000.000, se un termine appare in 1 documento allora avrebbe idf di 1.000.000, con log invece si smorza a 6.

Inoltre, se un termine appare in tutti i documenti -> df_t = N -> idf_t = log(N/N) = log(1) = 0 -> il termine non contribuisce affatto alla rilevanza di un documento rispetto alla query, come ci aspettiamo, mentre se avessimo lasciato senza log avremmo avuto idf_t = 1.

Vediamo dall'immagine questo nel confronto tra i due grafici con asse x il df e asse y l'idf: senza log l'idf passa da altissimo a bassissimo molto velocemente (già quando df è 10 idf diventa 1000). Con log invece la situazione è molto più stabile

<img src=img/idf_no_log.png width=400>

<img src=img/idf.png width=400>

Si osserva **idf ha un vero e proprio impatto solo per query con almeno due termini**: ad esempio nella query "arachnocentric line" l'idf è alto per arachnocentric (termine raro) e basso per line (termine frequente), quindi i documenti che contengono arachnocentric saranno considerati più rilevanti rispetto a quelli che contengono solo line. Al contrario, in una query con solo "arachnocentric", tutti i documenti considerati avranno lo stesso idf -> nel calcolo complessivo del tf-idf idf rappresenta solo una costante moltiplicativa che non influisce sull'ordinamento dei documenti. 

<img src=img/tfidf.png width=200>

Un'alternativa al df document frequency (numero di documenti in cui appare il termine) è il cf collection frequency (numero di volte che il termine appare nella collezione), tuttavia per il weighting è meglio df in quanto *ciò che interessa è in quanti documenti il termine è diffuso per sapere quanto è effettivamente raro nella collezione, non solo quante volte appare in totale*.

Definiamo quindi la formula definitiva del **tf-idf weighting**, moltiplicando tf e idf:
$$ w_{t, d} = (1 + \log(tf_{t, d})) \cdot \log \left( \frac{N}{df_t} \right) $$
**Quindi tf-idf aumenta all'aumentare della frequenza del termine nel documento e all'aumentare della rarità del termine nella collezione.**

### Vector Space Model
Dopo aver calcolato per ogni termine il suo peso tf-idf, possiamo modificare adeguatamente la matrice, ottenendo una **TF-IDF Matrix**. Quindi ogni documento diventa **un vettore di numeri reali**. Questo significa che possiamo pensare:
- un documento come un punto in uno spazio multidimensionale (ogni dimensione corrisponde a un termine)
- ogni termine come una coordinata di questo spazio, con valore pari al peso tf-idf del termine nel documento

Lo spazio di ogni documento è ad altissima dimensionalità (in R^|V|, dove |V| è il numero di termini distinti nella collezione), ma è anche molto sparso (la maggior parte dei termini non appare in un documento, quindi la maggior parte delle coordinate è 0). Possiamo applicare la stessa idea alla query: anch'essa può essere rappresentata come vettore di numeri reali della stessa dimensione in questo modo.

<img src=img/vector_space.png width=400>

A questo punto la cosa più intuitiva per capire la rilevanza di un documento rispetto a una query è **misurare la distanza tra i due vettori in quell'altissimo spazio vettoriale**: documenti più vicini alla query sono più rilevanti, mentre documenti più lontani sono meno rilevanti. 

#### Naive: Euclidean Distance
La prima idea è quella di usare la distanza euclidea, che misura la lunghezza del segmento che congiunge i due punti (query e documento) nello spazio vettoriale: 
$$ \text{euclidean-distance}(q, d) = \sqrt{\sum_{t \in V} (w_{t, q} - w_{t, d})^2} $$
**Tuttavia questa misura è problematica dal momento che influenzata pesantemente dalla lunghezza dei vettori**: due documenti possono avere distribuzione di termini molto simile, ma se uno è più lungo dell'altro (e quindi ha valori di tf-idf più alti) la distanza euclidea sarà comunque grande.

Immaginiamo ad esempio di prendere una query e un documento d ad essa molto simile. Sia ora d' lo stesso documento concatenato due volte: allora d' è rilevante quanto d rispetto alla query, tuttavia dal momento che ogni termine appare due volte in d' invece che una, i suoi pesi tf-idf saranno più alti e quindi la distanza euclidea tra query e d' sarà maggiore rispetto alla distanza tra query e d. **Cosa non cambia? La direzione dei due vettori!**

<img src=img/euclidean_distance.png width=400>

#### Cosine Similarity
Per confrontare gli angoli tra vettori la funzione più adatta è il coseno, in quanto *monotona decrescente rispetto all'angolo nell'intervallo rilevante*: a 0 gradi (massima similarity) corrisponde 1, a 90 gradi (nessuna similarity) corrisponde 0.

Per calcolare il coseno tra due angoli usiamo la cosine similarity, che deriva direttamente dalla formula di prodotto scalare tra due vettori:
$$ q \cdot d = ||q|| \cdot ||d|| \cdot \cos(\theta) $$
da cui si ricava:
$$ \text{cosine-similarity}(q, d) = \frac{q \cdot d}{||q|| \cdot ||d||} $$
dove si ricorda che la norma di un vettore è data dalla somma del quadrato di tutte le sue componenti (pesi tf-idf, in questo caso) tutto sotto radice:
$$ ||v|| = \sqrt{\sum_{t \in V} w_{t, v}^2} $$

Per rendere i calcoli più efficienti, l'idea è quella di **normalizzare i vettori** già dall'inizio, di modo che tutti abbiano norma 1 e che quindi la cosine similarity si riduca al semplice prodotto scalare:
$$ \text{cosine-similarity}(q, d) = q \cdot d = \sum_{t \in V} w_{t, q} \cdot w_{t, d} $$

Essendo tutti i vettori normalizzati, tutti sono al massimo di lunghezza 1 e quindi inscritti in un'ipersfera unitaria.

Esempio: Si calcola la cosine similarity tra tre romanzi. Si arriva alla conclusione che Sas è più simile a PaP rispetto a WH proprio perché WH contiene il termine wuthering che non appare in Sas e PaP, mentre SaS e PaP condividono gli altri termini in modo più marcato.

<p align="center">
<img src=img/cs1.png width=45%>
<img src=img/cs2.png width=50%>
</p>

#### Algoritmo per Cosine Score 
Di seguito vediamo l'algoritmo per calcolare il cosine score tra una query e un documento. Ci si ricollega all'inverted index, e si capisce dal punto 3. il perché questo è fondamentale per rendere il tutto più efficiente. L'idea è la seguente:
1. Si crea un array **Scores**, inizialmente tutto a 0, in cui Scores[d] rappresenta lo score del documento d rispetto alla query
2. Per ogni termine della query t, si calcola il suo peso tf-idf w_{t, q} e si accede alla posting list di t nell'inverted index. 
3. Nella posting list di t, per ogni documento d in cui appare t, si calcola il peso tf-idf w_{t, d} e si aggiorna lo score del documento d rispetto alla query sommando w_{t, q} * w_{t, d} a Scores[d]. *Stiamo sostanzialmente facendo il prodotto scalare tra i due vettori*, **ma in modo molto efficiente**: invece di dover scorrere tutte le componenti del vettore (che sono tantissime), ci limitiamo a scorrere solo i termini della query e i documenti in cui appaiono quei termini, che sono molti meno (si ricorda che i vettori sono molto sparsi)
4. Alla fine di questo processo, si divide Scored[d] per la lunghezza del documento d (norma del vettore del documento) per ottenere la cosine similarity finale tra query e documento d.
5. Si restituiscono i Top K documenti con i punteggi più alti.

<img src=img/algorithm.png width=400>

Questo algoritmo è **TAAT** (term-at-a-time) in quanto scorre i termini della query uno alla volta, aggiornando gli score dei documenti che li contengono di conseguenza. Tuttavia può anche essere adattato in modalità **DAAT** (document-at-a-time), in cui si scorre un documento alla volta, calcolando il suo score rispetto alla query prima di passare al documento successivo. 

Si osserva che memorizzare i pesi tf-idf w_{t, d} in ogni posting list potrebbe essere **molto costoso** in termini di spazio dal momento che i pesi sono numeri reali (float). Per questa ragione è invece preferibile memorizzare:
- la tf_{t, d} (term frequency) per ogni posting (che è un numero intero, si riduce lo spazio)
- la idf_t (inverse document frequency) soltanto in testa alla posting list di t, in modo da poterla recuperare quando serve per calcolare w_{t, d} = tf_{t, d} * idf_t. *Si noti che già dalla prima lezione avevamo anticipato come per ogni termine fosse necessario memorizzare la sua df, questo è un altro dei motivi fondamentali!*

Per prendere poi i Top K documenti, si possono sfruttare strutture dati come code con priorità.

#### Varianti di TF-IDF

Si osservano nell'immagine qui sotto alcune varianti di tf weighting:

<img src=img/variants.png width=400>

Si approfondiscono le più rilevanti:
- nella versione **natural**, si conta il numero grezzo di occorrenze del termine nel documento (raw frequency n(t,d)), senza applicare il logaritmo. Come già detto, questo non è l'ideale in quanto la rilevanza non cresce proporzionalmente con il numero di occorrenze, ma è comunque una variante possibile.
- **boolean tf**: vale 1 se il termine appare nel documento, 0 altrimenti. In questo modo si perde completamente l'informazione sulla frequenza dei termini, ma è una variante ancora più semplice da implementare e che può essere sufficiente in alcuni casi (ad esempio se la collezione è molto rumorosa e quindi la frequenza dei termini non è affidabile).

Le prossime due varianti sono più sofisticate e risolvono un problema importante della tf grezza (che risolveva anche il logaritmo), ovvero che se un termine compare molte volte in un documento allora il punteggio cresce tantissimo anche se in realtà l'importanza del documento non è proporzionale alla frequenza del termine.
- **Frac TF**: la formula è $tf_{frac}(t, d, k) = n(t, d) / (n(t, d) + k)$, dove n(t, d) è la frequenza del termine t nel documento d e k è un parametro che controlla quanto velocemente la funzione si satura. In pratica questa variante permette di limitare tf in un intervallo $[0, 1)$, infatti se n(t, d) = 0 allora tf_{frac} = 0, se n(t, d) aumenta tf_{frac} cresce ma tende al più ad 1! Es. con k = 1, se n(t, d) = 0 allora tf_{frac} = 0, se n(t, d) = 1 allora tf_{frac} = 1/2 = 0.5, se n(t, d) = 2 allora tf_{frac} = 2/3 ≈ 0.67, se n(t, d) = 10 allora tf_{frac} = 10/11 ≈ 0.91, se n(t,d) = 11 alla fine tf_{frac} = 11/12 ≈ 0.92 etc... **Il punto è che questa funzione quindi mostra quanto sia importante nel punteggio tf passare da 0 a 1 occorrenza, ma quasi irrilevante passare da 10 a 11 occorrenze! Chiaramente più k è grande più lenta è la crescita di tf_{frac} al crescere di n(t, d)**.
- **BM25 TF**: la formula è $tf_{bm25}(t, d, c, k, b) = \frac{n(t, d)}{n(t, d) + k \cdot (b \cdot ndl(d,c) + (1 - b))}$. Qui n(t,d) è la frequenza del termine t nel documento d, c è la collezione, ndl(d,c) è la lumghezza del documento normalizzata rispetto alla collezione mentre k e b sono parametri. In particolare $ndl(d,c) = \frac{N(d)}{adl(c)}$, dove N(d) è la lunghezza del documento d e adl(c) è la lunghezza media dei documenti nella collezione c. BM25 assomiglia a frac tf in quanto è anch'essa una funzione che cresce con n ma con rendimento decrescente. Ciò che introduce BM25 è la lunghezza del documento: **un documento lungo tende naturalmente a contenere più occorrenze di molti termini -> se non si corregge questo effetto, i documenti più lunghi sono favoriti ingiustamente**. BM25 risolve il problema grazie a ndl(d,c) al denominatore: **se un documento è più lungo della media, allora ndl(d,c) > 1 e quindi il denominatore è più grande -> tf_{bm25} è più piccolo rispetto a un documento di lunghezza minore ma con la stessa frequenza n(t, d). Al contrario, se un documento è più corto della media, allora ndl(d,c) < 1 e quindi il denominatore è più piccolo -> tf_{bm25} è più grande.** Riguardo i parametri, k come prima controlla la velocità di saturazione, mentre b controlla l'importanza della normalizzazione per la lunghezza del documento. Se b = 0 allora la formula torna a frac tf (no correzione per la lunghezza), se b = 1 allora la normalizzazione è massima. In generale, valori di b intorno a 0.75 sono considerati un buon compromesso.